## Multi-Agent Quiz Solver — PoliMillionaire
Pattern dal tutorial adattato per modelli locali via Ollama.

```
Orchestrator
  ├── @tool → MathAgent    (tools: calculator, python_repl, wolframalpha_simple)
  └── @tool → KnowledgeAgent (tools: wikipedia_search, duckduckgo_search)
```

**Requisiti**: `ollama pull qwen2.5:1.5b` (o modello preferito)

## 1 · Modello locale

In [68]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent

# Replace with 'qwen2.5-math:1.5b' for the math sub-agent if available
# Pulled from Ollama it must be: ollama pull qwen2.5:1.5b
def make_model(model_name: str = "qwen2.5:1.5b", temperature: float = 0.0):
    return ChatOllama(
        model=model_name,
        temperature=temperature,
    )

base_model = make_model()
print("The model is ready")

The model is ready


## 2 · Tool del MathAgent

In [69]:
import math
import re

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression safely.
    Supports: +, -, *, /, **, sqrt(), log(), sin(), cos(), tan(), pi, e.
    Input example: 'sqrt(144) + 3**2'
    """
    allowed = {
        "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
        "sin": math.sin,   "cos": math.cos, "tan": math.tan,
        "pi": math.pi,     "e": math.e,
        "abs": abs,        "round": round,  "pow": pow,
        "factorial": math.factorial, "gcd": math.gcd,
        "comb": math.comb, "perm": math.perm,
    }
    sanitized = re.sub(r"[^0-9\+\-\*\/\(\)\.\,\s\_a-zA-Z]", "", expression)
    try:
        result = eval(sanitized, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as ex:
        return f"Error: {ex}"


@tool
def modular_arithmetic(a: int, b: int, mod: int, operation: str = "multiply") -> str:
    """
    Perform modular arithmetic: (a OP b) mod n.
    Operations: 'add', 'subtract', 'multiply', 'power'.
    Example: a=12, b=16, mod=24, operation='multiply'  →  (12*16) mod 24 = 0
    """
    ops = {
        "add":      (a + b) % mod,
        "subtract": (a - b) % mod,
        "multiply": (a * b) % mod,
        "power":    pow(a, b, mod),  # fast modular exponentiation
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Use: add, subtract, multiply, power."
    return str(ops[operation])


@tool
def statistics_helper(data: str, operation: str) -> str:
    """
    Compute basic statistics on a comma-separated list of numbers.
    Operations: 'mean', 'median', 'std', 'variance', 'range', 'sort'.
    Input example: data='86,94,97,88,96', operation='median'
    """
    import statistics
    try:
        nums = [float(x.strip()) for x in data.split(",")]
    except ValueError:
        return "Error: provide comma-separated numbers."

    ops = {
        "mean":     lambda d: statistics.mean(d),
        "median":   lambda d: statistics.median(d),
        "std":      lambda d: statistics.stdev(d) if len(d) > 1 else 0,
        "variance": lambda d: statistics.variance(d) if len(d) > 1 else 0,
        "range":    lambda d: max(d) - min(d),
        "sort":     lambda d: sorted(d),
    }
    if operation not in ops:
        return f"Unknown operation. Use: {list(ops.keys())}"
    return str(ops[operation](nums))

print("Math tools loaded")

Math tools loaded


## 3 · Tool del KnowledgeAgent

In [70]:
@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return the summary of the most relevant article.
    Use for factual recall: history, science, geography, famous people, etc.
    Input example: query='French Revolution'
    """
    import wikipedia
    try:
        wikipedia.set_lang("en")
        results = wikipedia.search(query, results=3)
        if not results:
            return "No Wikipedia results found."
        # Try pages in order until one works
        for title in results:
            try:
                page = wikipedia.page(title, auto_suggest=False)
                # Return first 800 chars of summary to stay within context
                return page.summary[:800]
            except wikipedia.DisambiguationError as de:
                continue
        return "Could not retrieve a Wikipedia page."
    except Exception as ex:
        return f"Wikipedia error: {ex}"


@tool
def duckduckgo_search(query: str) -> str:
    """
    Search the web via DuckDuckGo and return top 3 text snippets.
    Use for recent facts or when Wikipedia is insufficient.
    Returns raw non-generated content (text snippets only — no LLM).
    """
    from duckduckgo_search import DDGS
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."
        return "\n\n".join(
            f"[{r['title']}]\n{r['body']}" for r in results
        )
    except Exception as ex:
        return f"Search error: {ex}"

print("Knowledge tools loaded")

Knowledge tools loaded


## 4 · Sub-agents

In [ ]:
# LangGraph's create_react_agent handles the ReAct loop internally.

# ── MathAgent ────────────────────────────────────────────────────────────────
math_tools = [calculator, modular_arithmetic, statistics_helper]

math_agent = create_agent(
    model=make_model(temperature=0.0),
    tools=math_tools,
    debug=True,
)

# ── KnowledgeAgent ───────────────────────────────────────────────────────────
knowledge_tools = [wikipedia_search, duckduckgo_search]

knowledge_agent = create_agent(
    model=make_model(temperature=0.0),
    tools=knowledge_tools,
    debug=True,
)

print("Sub-agents created")

Sub-agents created, they are.


## 5 · Orchestrator

In [ ]:
# ── Wrapping sub-agents come @tool ───────────────────────────────────────────

@tool
def ask_math_agent(question_with_choices: str) -> str:
    """
    Delegate a mathematical question to the Math specialist agent.
    Use for: arithmetic, algebra, calculus, statistics, probability,
    modular arithmetic, combinatorics, geometry.
    Pass the full question text and all answer choices.
    """
    result = math_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 12},
    )
    return result["messages"][-1].content


@tool
def ask_knowledge_agent(question_with_choices: str) -> str:
    """
    Delegate a factual knowledge question to the Knowledge specialist agent.
    Use for: history, geography, science, literature, general culture,
    current events, famous people, definitions.
    Pass the full question text and all answer choices.
    """
    result = knowledge_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 8},
    )
    return result["messages"][-1].content


# ── Orchestrator ──────────────────────────────────────────────────────────────
orchestrator_tools = [ask_math_agent, ask_knowledge_agent]

orchestrator = create_agent(
    model=make_model(temperature=0.0),
    tools=orchestrator_tools,
    debug=True,
)

print("Orchestrator ready")

Orchestrator ready


## 6 · Wrapper per il quiz

In [ ]:
import re
def format_question(question: str, choices: dict) -> str:
    """Format question + choices for the orchestrator."""
    lines = [question, ""]
    for letter, text in choices.items():
        lines.append(f"{letter}: {text}")
    lines.append("\nWhich answer is correct? Reply with just the letter.")
    return "\n".join(lines)


def extract_letter(raw: str) -> str:
    """Extract A/B/C/D from agent output."""
    m = re.search(r"\b([A-Da-d])\b", raw.strip())
    return m.group(1).upper() if m else "A"


def answer_quiz_question(question: str, choices: dict) -> dict:
    """
    Entry point per il quiz.

    Parameters
    ----------
    question : str  — testo della domanda
    choices  : dict — {'A': '...', 'B': '...', 'C': '...', 'D': '...'}

    Returns
    -------
    {'answer_letter': 'B', 'answer_text': '...', 'raw': '...'}
    """
    prompt = ORCHESTRATOR_SYSTEM + "\n\n" + format_question(question, choices)
    result = orchestrator.invoke(
        {"messages": [("human", prompt)]},
        config={"recursion_limit": 8},
    )
    raw    = result["messages"][-1].content
    letter = extract_letter(raw)
    return {
        "answer_letter": letter,
        "answer_text":   choices.get(letter, "?"),
        "raw":           raw,
    }

## 7 · Test

In [ ]:
# ── Test 1: math (dal log degli errori reali) ─────────────────────────────────
q1 = "What is the units digit in the standard decimal expansion of 7^25?"
c1 = {"A": "5", "B": "7", "C": "3", "D": "1"}
r1 = answer_quiz_question(q1, c1)
print(f"\n[MATH] Answer: {r1['answer_letter']}) {r1['answer_text']}")  # expected: B

[values] {'messages': [HumanMessage(content="\nYou are a quiz coordinator for the game 'Who wants to be a PoliMillionaire?'.\n\nGiven a question and four choices (A/B/C/D):\n1. Decide whether it needs mathematical computation or factual knowledge.\n2. Delegate to the correct specialist:\n   - Mathematical (arithmetic, stats, algebra, logic) → use ask_math_agent\n   - Factual/general knowledge → use ask_knowledge_agent\n3. Use the specialist's answer to pick the correct letter.\n\n\n\nWhat is the units digit in the standard decimal expansion of 7^25?\n\nA) 5\nB) 7\nC) 3\nD) 1\n\nWhich answer is correct? Reply with just the letter.", additional_kwargs={}, response_metadata={}, id='1b3f624a-122d-4c38-a07a-3c4b7c6d8e62')]}
[updates] {'model': {'messages': [AIMessage(content="To determine the units digit in the standard decimal expansion of \\(7^{25}\\), we need to look at the pattern of the units digits for powers of 7.\n\nLet's list out the first few powers of 7 and observe their units di

In [75]:
# ── Test 2: statistiche (altra categoria problematica) ────────────────────────
q2 = (
    "On her previous five attempts Sarah had achieved times, in seconds, of "
    "86, 94, 97, 88 and 96. After her sixth try she brought her median time "
    "down to 92 seconds. What was her time for her sixth attempt?"
)
c2 = {"A": "92", "B": "90", "C": "87", "D": "10"}
r2 = answer_quiz_question(q2, c2)
print(f"\n[STATS] Answer: {r2['answer_letter']}) {r2['answer_text']}")  # expected: B

[values] {'messages': [HumanMessage(content="\nYou are a quiz coordinator for the game 'Who wants to be a PoliMillionaire?'.\n\nGiven a question and four choices (A/B/C/D):\n1. Decide whether it needs mathematical computation or factual knowledge.\n2. Delegate to the correct specialist:\n   - Mathematical (arithmetic, stats, algebra, logic) → use ask_math_agent\n   - Factual/general knowledge → use ask_knowledge_agent\n3. Use the specialist's answer to pick the correct letter.\n\n\n\nOn her previous five attempts Sarah had achieved times, in seconds, of 86, 94, 97, 88 and 96. After her sixth try she brought her median time down to 92 seconds. What was her time for her sixth attempt?\n\nA) 92\nB) 90\nC) 87\nD) 10\n\nWhich answer is correct? Reply with just the letter.", additional_kwargs={}, response_metadata={}, id='31d4b21f-2b4b-4b4a-8709-08459b6fa5e0')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_

In [76]:
# ── Test 3: knowledge ─────────────────────────────────────────────────────────
q3 = "Which planet is known as the Red Planet?"
c3 = {"A": "Venus", "B": "Jupiter", "C": "Mars", "D": "Saturn"}
r3 = answer_quiz_question(q3, c3)
print(f"\n[KNOWLEDGE] Answer: {r3['answer_letter']}) {r3['answer_text']}")  # expected: C

[values] {'messages': [HumanMessage(content="\nYou are a quiz coordinator for the game 'Who wants to be a PoliMillionaire?'.\n\nGiven a question and four choices (A/B/C/D):\n1. Decide whether it needs mathematical computation or factual knowledge.\n2. Delegate to the correct specialist:\n   - Mathematical (arithmetic, stats, algebra, logic) → use ask_math_agent\n   - Factual/general knowledge → use ask_knowledge_agent\n3. Use the specialist's answer to pick the correct letter.\n\n\n\nWhich planet is known as the Red Planet?\n\nA) Venus\nB) Jupiter\nC) Mars\nD) Saturn\n\nWhich answer is correct? Reply with just the letter.", additional_kwargs={}, response_metadata={}, id='3d2671f7-d464-4d95-a8ed-2267c73855f4')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_knowledge_agent", "arguments": {"question_with_choices": "Which planet is known as the Red Planet?\\n\\nA) Venus\\nB) Jupiter\\nC) Mars\\nD) Saturn"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:

## 8 · Integrazione con l'API del quiz

In [77]:
import os
import dotenv
from millionaire_client import MillionaireClient

dotenv.load_dotenv()

API_URL  = "http://131.175.15.22:51111/"
username = os.getenv("USER")
pwd      = os.getenv("PASSWORD")

client = MillionaireClient(API_URL)
client.login(username, pwd)
print("Logged in.")

comps = client.competitions.list_all()
print("Competitions:", [c.name for c in comps])

LETTERS = ["A", "B", "C", "D"]

for comp in comps:
    print(f"\n{'='*60}")
    print(f"Competition: {comp.name}")

    session = client.game.start(comp.id)

    while session.in_progress:
        question = session.current_question
        if question is None:
            break

        # Map options list → {'A': text, 'B': text, ...}
        choices = {LETTERS[i]: opt.text for i, opt in enumerate(question.options)}

        result = answer_quiz_question(question.text, choices)

        # Find the option object for the chosen letter
        chosen_idx = LETTERS.index(result["answer_letter"])
        chosen_opt = question.options[chosen_idx]

        answer = session.answer(chosen_opt.id)
        status = "OK" if answer.correct else "WRONG"
        print(f"  L{question.level}: {result['answer_letter']}) {result['answer_text']} -> {status}")

    print(f"  Earned: {session.earned_amount}")

Logged in.
Competitions: ['Entertainment', 'Ancient History and Politics', 'Science and Nature', 'Maths']

Competition: Entertainment
[values] {'messages': [HumanMessage(content="\nYou are a quiz coordinator for the game 'Who wants to be a PoliMillionaire?'.\n\nGiven a question and four choices (A/B/C/D):\n1. Decide whether it needs mathematical computation or factual knowledge.\n2. Delegate to the correct specialist:\n   - Mathematical (arithmetic, stats, algebra, logic) → use ask_math_agent\n   - Factual/general knowledge → use ask_knowledge_agent\n3. Use the specialist's answer to pick the correct letter.\n\n\n\nWhat is the fundamental principle of the plot in Casablanca?\n\nA) Romantic love\nB) Action and adventure\nC) Political intrigue and sacrifice\nD) Comedy and humor\n\nWhich answer is correct? Reply with just the letter.", additional_kwargs={}, response_metadata={}, id='b8dff563-d760-4d44-8b59-d729f2cf0c85')]}
[updates] {'model': {'messages': [AIMessage(content='', additional